# Phân tích dữ liệu của 71 file scan sau OCR

Notebook **chỉ đọc và chỉ ra vấn đề**, không sửa dữ liệu. Mọi số liệu do code sinh ra khi chạy.

**Nguyên tắc phân tích (mỗi mục):** 1. Tổng quan → 2. Lỗi / vấn đề (có ví dụ thật) → 3. Tổng hợp số lượng → 4. Hướng giải quyết.

**Về cách đo:** notebook này **không dùng điểm tin cậy của Tesseract** làm thước đo chất lượng. Đó là điểm máy tự chấm, không giải thích được là tỷ lệ chữ đúng. Thay vào đó chỉ dùng các kiểm tra mà cách tính và giới hạn đều nêu được:

| Mục | Kiểm tra | Cơ sở |
|---|---|---|
| 2 | Trang gần như không có chữ | đếm từ trên trang |
| 3 | Ký tự lạ, dòng rác | ký tự ngoài chữ / số / dấu câu thường |
| 4 | Từ không có trong từ vựng chuẩn; gợi ý sửa dấu | so với chữ từ PDF có lớp chữ + nội dung website |
| 5 | Tên môn học trên website có trong chữ OCR không | đối chiếu với **nguồn độc lập** (không phải do OCR sinh ra) |
| 6 | Trang có dạng bảng | tỷ lệ dòng ngắn |
| 7 | Trùng lặp | so sánh nội dung |

Đầu vào: `extract/text/` (chữ từng file), `extract/ocr/` (chữ từng trang), `extract/manifest.csv`, `data/crawl_v2.json`.

In [1]:
import sys, re, json, unicodedata, hashlib
sys.path.insert(0, "..")
from collections import Counter, defaultdict
import pandas as pd
import extract_files as ef
import ocr_scans as oc

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 170)

issues = []
def add_issue(muc, loi, so_luong, don_vi, huong):
    issues.append({"mục": muc, "lỗi / vấn đề": loi, "số lượng": int(so_luong), "đơn vị": don_vi, "hướng giải quyết": huong})

def show(d):
    print(d.to_string())

def norm_ws(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", s)).strip()

def strip_marks(s):
    d = unicodedata.normalize("NFD", s.replace("đ", "d").replace("Đ", "D"))
    return "".join(c for c in d if not unicodedata.combining(c))

def words(s):
    return re.findall(r"[^\W\d_]+", unicodedata.normalize("NFC", s).lower())

ex = {r["file_key"]: r for r in ef.read_extract_log()}
mf = {r["file_key"]: r for r in ef.load_manifest()}
ocr_keys = sorted(k for k, r in ex.items() if r["method"] == "pdf_ocr" and r["status"] == "ok")

def doc_type(name):
    n = strip_marks(name).lower()
    if "ban_mo_ta" in n or "ban mo ta" in n: return "Bản mô tả CTĐT"
    if n.startswith("ctdt") or "_ctdt" in n: return "Chương trình đào tạo (CTĐT)"
    if "phu_luc" in n or "phu luc" in n: return "Phụ lục"
    if n.startswith("qd") or "quyet dinh" in n or "1. qd" in n: return "Quyết định"
    return "Khác (tên học phần, văn bản chính, ...)"

rows = []
for k in ocr_keys:
    m = mf.get(k, {})
    t = (ef.TEXT / f"{k}.txt").read_text(encoding="utf-8")
    rows.append({"file_key": k, "record_id": m.get("record_id", ""), "category": m.get("category", ""),
                 "name": m.get("link_name", ""), "doc_type": doc_type(m.get("link_name", "")),
                 "n_pages": int(ex[k]["n_pages"]), "n_chars": len(t), "n_lines": t.count("\n") + 1, "text": t})
files = pd.DataFrame(rows)

prow = []
for k in ocr_keys:
    for p in sorted((oc.ocr_dir() / k).glob("p*.json")):
        d = json.loads(p.read_text(encoding="utf-8"))
        prow.append({"file_key": k, "page": d["page"], "rotate": d.get("rotate", 0), "text": d["text"],
                     "n_words": len(words(d["text"]))})
pages = pd.DataFrame(prow)
pages["record_id"] = pages["file_key"].map(lambda k: mf.get(k, {}).get("record_id", ""))
print(f"Đã nạp {len(files)} file OCR và {len(pages)} trang")

Đã nạp 71 file OCR và 2045 trang


## 1. Tổng quan

In [2]:
print("1. TỔNG QUAN")
print(f"   File OCR: {len(files)} | trang: {len(pages)} | ký tự: {files['n_chars'].sum():,} | dòng: {files['n_lines'].sum():,}")
print()
print("   a) Theo bản ghi (thông báo / quyết định chứa file)")
show(files.groupby(["record_id", "category"]).agg(so_file=("file_key", "size"), so_trang=("n_pages", "sum"), so_ky_tu=("n_chars", "sum")))
print()
print("   b) Theo loại tài liệu (phân loại theo TÊN FILE, chỉ mang tính tham khảo)")
show(files.groupby("doc_type").agg(so_file=("file_key", "size"), so_trang=("n_pages", "sum"), ky_tu_trung_vi=("n_chars", "median")))
print()
print("   c) Độ dài mỗi trang (ký tự): ", pages["text"].str.len().quantile([0, .25, .5, .75, 1]).astype(int).tolist(), "(min, 25%, trung vị, 75%, max)")
print()
print("   d) Trang đã tự xoay trước khi OCR (trang bị lật / nằm ngang trong bản scan)")
rot = pages["rotate"].value_counts().rename("số trang").to_frame()
rot.index = rot.index.map(lambda a: "không xoay" if a == 0 else f"xoay {a}°")
show(rot)
print()
print("2. LỖI: không có lỗi ở mức tổng quan")
print("3. TỔNG HỢP: 71/71 file đều có chữ")

1. TỔNG QUAN
   File OCR: 71 | trang: 2045 | ký tự: 3,147,445 | dòng: 122,652

   a) Theo bản ghi (thông báo / quyết định chứa file)
                                so_file  so_trang  so_ky_tu
record_id category                                         
tb-2223   hoc_phi                     1         2      2569
tb-2234   chuong_trinh_dao_tao       47      1627   2555095
tb-2235   chuong_trinh_dao_tao       20       361    453714
tb-2248   hoc_phi                     1         5      4991
tb-2281   quy_che                     1        46    122928
tb-2284   ke_hoach_dao_tao            1         4      8148

   b) Theo loại tài liệu (phân loại theo TÊN FILE, chỉ mang tính tham khảo)
                                         so_file  so_trang  ky_tu_trung_vi
doc_type                                                                  
Bản mô tả CTĐT                                20      1400        101988.0
Chương trình đào tạo (CTĐT)                   27       227         15365.0
Khác (tên 

**Giải thích mục d:** nhiều trang chương trình đào tạo là bảng ngang được quét xoay 90°. Chương trình nhận hướng trang bằng chức năng OSD của Tesseract rồi xoay lại trước khi OCR. Nếu không xoay, chữ bị đọc ngược và vô nghĩa.

**4. Hướng giải quyết:** không có lỗi. Phân loại theo tên file chỉ dùng tham khảo; cần đối chiếu nội dung nếu muốn gắn nhãn chính xác.

## 2. Trang gần như không có chữ

In [3]:
print("1. TỔNG QUAN: số từ trên mỗi trang")
print("   phân vị (min, 25%, trung vị, 75%, max):", pages["n_words"].quantile([0, .25, .5, .75, 1]).astype(int).tolist())
LOW_WORDS = 30
few = pages[pages["n_words"] < LOW_WORDS]
print()
print(f"2. LỖI / VẤN ĐỀ: trang có dưới {LOW_WORDS} từ = {len(few)} / {len(pages)}")
show(few.assign(file_key=few["file_key"].str.slice(0, 30), text=few["text"].str.replace(r"\s+", " ", regex=True).str.slice(0, 60))
        [["record_id", "file_key", "page", "rotate", "n_words", "text"]])
print()
print("3. TỔNG HỢP:")
print(f"   {len(few)} trang | {few['file_key'].nunique()} file | số trang rỗng hoàn toàn: {int((pages['n_words'] == 0).sum())}")
add_issue("2. Trang ít chữ", f"trang có dưới {LOW_WORDS} từ", len(few), "trang", "xem ảnh trang: trang bìa / trang trắng / chỉ con dấu thì bỏ; trang có nội dung thì OCR lại")

1. TỔNG QUAN: số từ trên mỗi trang
   phân vị (min, 25%, trung vị, 75%, max): [0, 194, 287, 411, 1001]

2. LỖI / VẤN ĐỀ: trang có dưới 30 từ = 16 / 2045
     record_id                        file_key  page  rotate  n_words                                                          text
231    tb-2235  drive-file_14mmWw9rAR68IFzgrMo     7       0       28  Đánh giá Bài thi cuối kỳ kết thúc Thi trắc „| CLOI | 50% The
327    tb-2234  drive-file_196IJrjAFyY-aKHJ6P3    53       0       13  20. Thời điểm thiết kế: ngày 16 tháng 0ó năm 2026 LUÁ FS: Tr
640    tb-2234  drive-file_1G7U1LgT03GvIt2j-P7    30     180        1                                                             ũ
664    tb-2234  drive-file_1G7U1LgT03GvIt2j-P7    54       0        0                                                              
674    tb-2234  drive-file_1G7U1LgT03GvIt2j-P7    64       0        4                                        ` ! 20H !AO ZuØ, . F -
696    tb-2235  drive-file_1KWafD7bhtbzzYHZtvL    14   

**Giải thích:** trang ít chữ thường là trang bìa, trang trắng hoặc trang chỉ có con dấu; cũng có thể là trang có nội dung nhưng OCR không đọc được. Cột `text` ở trên cho thấy OCR đọc ra gì để phân biệt hai trường hợp.

**4. Hướng giải quyết:** xem ảnh các trang này một lần. Trang không có nội dung thì loại khỏi corpus; trang có nội dung mà không đọc được thì OCR lại với cài đặt khác (dpi cao hơn, chế độ phân đoạn khác).

## 3. Ký tự lạ và dòng rác

In [4]:
ASCII_PUNCT = set(" .,;:!?-_()[]{}/\\\"'%&+*=#@<>|~^$`")
OK_EXTRA = set("“”‘’–—…•°²³")

def ok_char(c):
    return c.isalnum() or c.isspace() or c in ASCII_PUNCT or c in OK_EXTRA

odd = Counter()
odd_files = defaultdict(set)
for k, t in zip(files["file_key"], files["text"]):
    for c in t:
        if not ok_char(c):
            odd[c] += 1
            odd_files[c].add(k)

print("1. TỔNG QUAN: ký tự KHÔNG phải chữ / số / dấu cách / dấu câu thường (chữ Việt có dấu được coi là hợp lệ)")
rows = [(repr(c), f"U+{ord(c):04X}", unicodedata.name(c, "?"), n, len(odd_files[c])) for c, n in odd.most_common(15)]
show(pd.DataFrame(rows, columns=["ký tự", "mã", "tên", "số lần", "số file"]))

def short_number_line(l):
    """Dòng ngắn có chữ số (vd '39', '2.'): có thể là STT / số tín chỉ trong bảng nên KHÔNG tính là rác."""
    s = re.sub(r"\s", "", l)
    return 0 < len(s) <= 3 and any(c.isdigit() for c in s)

def noise_line(l):
    s = l.strip()
    if not s or short_number_line(s):
        return False
    nospace = re.sub(r"\s", "", s)
    good = sum(1 for c in nospace if c.isalnum())
    return len(nospace) <= 2 or good / len(nospace) < 0.5

all_lines = [(k, l) for k, t in zip(files["file_key"], files["text"]) for l in t.split("\n") if l.strip()]
noise = [(k, l) for k, l in all_lines if noise_line(l)]
kept_short = sum(1 for k, l in all_lines if short_number_line(l))
print()
print("2. LỖI / VẤN ĐỀ — dòng rác (dòng ≤ 2 ký tự không có chữ số, hoặc dưới 50% là chữ / số). Ví dụ thật:")
for k, l in noise[::max(1, len(noise) // 8)][:8]:
    print(f"   {mf[k]['record_id']} | {l.strip()[:60]!r}")
print()
print("3. TỔNG HỢP")
tot_odd = sum(odd.values())
print(f"   Ký tự lạ: {tot_odd:,} lần, {len(odd)} loại khác nhau, chiếm {100 * tot_odd / files['n_chars'].sum():.2f}% số ký tự")
print(f"   Dòng rác: {len(noise):,} / {len(all_lines):,} dòng không trống ({100 * len(noise) / len(all_lines):.1f}%)")
print(f"   Không tính là rác: {kept_short:,} dòng ngắn chỉ có số (có thể là STT, số tín chỉ trong bảng)")
add_issue("3. Ký tự lạ / dòng rác", "ký tự lạ (ngoài chữ, số, dấu câu thường)", tot_odd, "ký tự", "xóa ký tự lạ vô nghĩa; giữ ký hiệu có nghĩa (≥, ≤, ×) sau khi rà danh sách")
add_issue("3. Ký tự lạ / dòng rác", "dòng rác (≤ 2 ký tự không có số, hoặc < 50% là chữ / số)", len(noise), "dòng", "xóa dòng rác khi chuẩn hóa; giữ dòng ngắn chỉ có số")

1. TỔNG QUAN: ký tự KHÔNG phải chữ / số / dấu cách / dấu câu thường (chữ Việt có dấu được coi là hợp lệ)
   ký tự      mã                                         tên  số lần  số file
0    '¡'  U+00A1                   INVERTED EXCLAMATION MARK    1116       46
1    '¬'  U+00AC                                    NOT SIGN    1110       62
2    '„'  U+201E                 DOUBLE LOW-9 QUOTATION MARK     597       55
3    '⁄'  U+2044                              FRACTION SLASH     208       59
4    '†'  U+2020                                      DAGGER     107       24
5    '‹'  U+2039   SINGLE LEFT-POINTING ANGLE QUOTATION MARK      96       32
6    '®'  U+00AE                             REGISTERED SIGN      81       29
7    '»'  U+00BB  RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK      74       33
8    '§'  U+00A7                                SECTION SIGN      70       30
9    '›'  U+203A  SINGLE RIGHT-POINTING ANGLE QUOTATION MARK      59       30
10   '©'  U+00A9                     

**Giải thích:** ký tự lạ và dòng rác đến từ con dấu, chữ ký, đường kẻ bảng và vết bẩn của bản scan, bị OCR đọc thành ký hiệu. Quy tắc phân loại dựa vào tỷ lệ chữ / số trong dòng. **Dòng ngắn chỉ có số** (ví dụ `39`, `2.`) không tính là rác vì có thể là số thứ tự hoặc số tín chỉ trong bảng; nếu tính vào thì sẽ xóa nhầm dữ liệu thật.

**4. Hướng giải quyết:** xóa dòng rác bằng quy tắc trên nhưng rà trước danh sách ký tự lạ ở bảng mục 1 (ví dụ `≥`, `≤` có nghĩa trong bảng điểm, không được xóa).

## 4. Từ không có trong từ vựng chuẩn và gợi ý sửa dấu

**Cách làm:** lập từ vựng chuẩn từ hai nguồn *không đi qua OCR*: chữ trích từ PDF có lớp chữ và nội dung website (`crawl_v2.json`). Từ trong chữ OCR mà **chưa từng xuất hiện** trong từ vựng này gọi là *từ lạ*. Từ lạ là dấu hiệu gián tiếp của lỗi OCR.

**Giới hạn:** tên riêng, thuật ngữ chuyên ngành và tên môn học mới cũng bị tính là từ lạ dù không sai. Để biết mức này lớn đến đâu, notebook đo thêm tỷ lệ từ lạ của **chính chữ chuẩn** khi bỏ riêng từng file khỏi từ vựng (đối chứng).

In [5]:
clean_files = {k: (ef.TEXT / f"{k}.txt").read_text(encoding="utf-8") for k, r in ex.items()
               if r["method"] == "pdf_text" and r["status"] == "ok"}
recs = json.loads(ef.SRC.read_text(encoding="utf-8"))
web_text = "\n".join(r["title"] + "\n" + r["content"] for r in recs)

vocab = Counter()
per_file = {}
for k, t in clean_files.items():
    c = Counter(words(t)); per_file[k] = c; vocab.update(c)
web_c = Counter(words(web_text)); vocab.update(web_c)

print("1. TỔNG QUAN")
print(f"   Từ vựng chuẩn: {len(vocab):,} từ khác nhau, {sum(vocab.values()):,} lượt từ")
print(f"   Nguồn: {len(clean_files)} PDF có lớp chữ + nội dung website ({len(recs)} bản ghi)")

# đối chứng: tỷ lệ từ lạ của chữ chuẩn khi bỏ riêng từng file khỏi từ vựng
base_tok = base_oov = 0
for k, c in per_file.items():
    for w, n in c.items():
        base_tok += n
        if vocab[w] - n <= 0:
            base_oov += n
base_rate = 100 * base_oov / base_tok

ocr_vocab = {}
tok_total = tok_oov = 0
oov_counter = Counter()
file_rates = []
for k, t in zip(files["file_key"], files["text"]):
    ws = words(t)
    o = [w for w in ws if w not in vocab]
    tok_total += len(ws); tok_oov += len(o); oov_counter.update(o)
    file_rates.append(100 * len(o) / max(len(ws), 1))
files["oov_%"] = file_rates
ocr_rate = 100 * tok_oov / tok_total

print()
print("2. LỖI / VẤN ĐỀ")
print(f"   Đối chứng (chữ chuẩn, bỏ riêng từng file khỏi từ vựng): {base_rate:.1f}% lượt từ là từ lạ")
print(f"   Chữ OCR: {ocr_rate:.1f}% lượt từ là từ lạ ({tok_oov:,} / {tok_total:,}), {len(oov_counter):,} từ lạ khác nhau")
print()
print("   Phân bố tỷ lệ từ lạ theo file OCR (%):", files["oov_%"].quantile([0, .25, .5, .75, 1]).round(1).tolist(), "(min, 25%, trung vị, 75%, max)")
print()
print("   15 từ lạ xuất hiện nhiều nhất trong chữ OCR:")
print("   ", [(w, n) for w, n in oov_counter.most_common(15)])

1. TỔNG QUAN
   Từ vựng chuẩn: 2,610 từ khác nhau, 137,697 lượt từ
   Nguồn: 75 PDF có lớp chữ + nội dung website (141 bản ghi)

2. LỖI / VẤN ĐỀ
   Đối chứng (chữ chuẩn, bỏ riêng từng file khỏi từ vựng): 5.1% lượt từ là từ lạ
   Chữ OCR: 7.3% lượt từ là từ lạ (45,476 / 622,663), 8,671 từ lạ khác nhau

   Phân bố tỷ lệ từ lạ theo file OCR (%): [1.7, 5.5, 6.5, 8.9, 17.4] (min, 25%, trung vị, 75%, max)

   15 từ lạ xuất hiện nhiều nhất trong chữ OCR:
    [('plo', 2209), ('pi', 1247), ('clo', 911), ('ñ', 907), ('cđr', 707), ('ˆ', 579), ('mm', 533), ('po', 498), ('tê', 477), ('sô', 459), ('ị', 454), ('li', 414), ('thê', 328), ('z', 302), ('ì', 300)]


In [6]:
# Gợi ý sửa: từ lạ w trùng với một từ chuẩn v khi bỏ hết dấu (cùng chữ cái gốc), v phổ biến hơn hẳn w
by_base = defaultdict(list)
for v, n in vocab.items():
    by_base[strip_marks(v)].append((v, n))
sugg = {}
for w, n in oov_counter.items():
    if len(w) < 3:
        continue
    cands = [(v, c) for v, c in by_base.get(strip_marks(w), []) if v != w and c >= 5 and c >= 3 * n]
    if cands:
        sugg[w] = max(cands, key=lambda x: x[1])

fixable = sum(oov_counter[w] for w in sugg)
print("3. GỢI Ý SỬA DẤU")
print(f"   Từ lạ CÓ GỢI Ý đổi dấu (chưa duyệt, một phần gợi ý là sai): {len(sugg):,} từ, {fixable:,} lượt ({100 * fixable / tok_oov:.1f}% lượt từ lạ)")
print(f"   Từ lạ còn lại (viết tắt, tên riêng, thuật ngữ, hoặc lỗi không phải lỗi dấu): {tok_oov - fixable:,} lượt")
print()
print("   20 gợi ý phổ biến nhất (sai → gợi ý, số lần):")
top = sorted(sugg.items(), key=lambda kv: -oov_counter[kv[0]])[:20]
show(pd.DataFrame([(w, oov_counter[w], v, c) for w, (v, c) in top], columns=["từ trong OCR", "số lần", "gợi ý", "số lần ở từ vựng chuẩn"]))
print()
print("   Ngữ cảnh thật của 3 gợi ý đầu (để kiểm tra bằng mắt gợi ý có đúng không):")
for w, (v, c) in top[:3]:
    for t in files["text"]:
        m = re.search(r"(?i)\b.{0,25}\b" + re.escape(w) + r"\b.{0,25}", t.replace("\n", " "))
        if m:
            print(f"   {w!r} → {v!r}: ...{m.group(0)}...")
            break
add_issue("4. Từ vựng", "lượt từ không có trong từ vựng chuẩn (gồm cả tên riêng, thuật ngữ)", tok_oov, "lượt từ", "phân loại: sửa dấu theo gợi ý; còn lại đối chiếu danh sách tên môn / tên riêng")
add_issue("4. Từ vựng", "trong đó có gợi ý đổi dấu (chưa duyệt; một phần gợi ý sai)", fixable, "lượt từ", "duyệt tay danh sách gợi ý rồi mới áp dụng; ưu tiên từ xuất hiện nhiều")

3. GỢI Ý SỬA DẤU
   Từ lạ CÓ GỢI Ý đổi dấu (chưa duyệt, một phần gợi ý là sai): 753 từ, 4,473 lượt (9.8% lượt từ lạ)
   Từ lạ còn lại (viết tắt, tên riêng, thuật ngữ, hoặc lỗi không phải lỗi dấu): 41,003 lượt

   20 gợi ý phổ biến nhất (sai → gợi ý, số lần):
   từ trong OCR  số lần   gợi ý  số lần ở từ vựng chuẩn
0         tiêng     259   tiếng                    1285
1           tôt     196     tốt                     607
2          quôc     190    quốc                    1014
3          điêm     159    điểm                     572
4          điêu     113    điều                     421
5          phằn      96    phần                     911
6           kêt      90     kết                     384
7          tỉnh      83    tính                     421
8           đên      77     đến                     271
9           đảo      74     đào                     731
10         củng      61    cung                     205
11          câp      57     cấp                     560
12       thượ

**Giải thích cách đọc:** so sánh tỷ lệ từ lạ của chữ OCR với tỷ lệ của chữ chuẩn (đối chứng) cho biết phần *vượt thêm* do OCR, không tính phần do tên riêng và thuật ngữ vốn không có trong từ vựng.

**Phải đọc kết quả thận trọng, vì ba lý do nhìn thấy ngay trong kết quả ở trên:**
1. *Từ lạ nhiều nhất không phải lỗi OCR.* `plo`, `clo`, `cđr`, `po`, `pi` là viết tắt của chương trình đào tạo (chuẩn đầu ra, mục tiêu) nên xuất hiện rất nhiều trong tài liệu scan nhưng ít trong từ vựng chuẩn. Phần "vượt thêm so với đối chứng" vì thế bị đẩy lên bởi từ vựng chuyên ngành, không hẳn do lỗi.
2. *Từ vựng chuẩn nhỏ* (khoảng 2,6 nghìn từ khác nhau) nên nhiều từ đúng nhưng hiếm cũng bị coi là lạ. Hệ quả là một số gợi ý **sai rõ ràng**, nhìn thấy trong bảng 20 gợi ý (ví dụ `tỉnh → tính`, `đảo → đào`, `thượng → thương`, `đói → đổi`).
3. Ngược lại nhiều gợi ý **đúng là lỗi dấu OCR** (`tiêng → tiếng`, `tôt → tốt`, `quôc → quốc`, `điêu → điều`), kiểm chứng được bằng ngữ cảnh in ở trên.

**Gợi ý sửa dấu chỉ là gợi ý:** quy tắc là *cùng chữ cái gốc, từ chuẩn phổ biến hơn từ sai ít nhất 3 lần và xuất hiện ≥ 5 lần*. Vì có gợi ý sai, **con số "có gợi ý" không phải số lỗi đã sửa đúng**; chỉ danh sách đã duyệt tay mới đáng tin.

**4. Hướng giải quyết:** (1) duyệt tay bảng gợi ý, giữ các cặp đúng thành từ điển sửa lỗi; (2) áp dụng từ điển cho toàn bộ chữ OCR; (3) phần từ lạ còn lại đối chiếu với danh sách tên môn học và tên riêng, không sửa tự động.

## 5. Đối chiếu tên môn học với nguồn độc lập

**Ý tưởng:** trang ngành trên website có danh sách tên môn học (trường `curriculum`) *không đi qua OCR*. Chương trình đào tạo trong các PDF scan chắc chắn nhắc đến các môn học này. Nếu tên môn có trong chữ OCR thì OCR đã đọc được phần đó.

**Ba mức khớp** (chuẩn hóa chữ thường, gộp khoảng trắng):
- *Khớp nguyên văn*: tên môn xuất hiện đúng từng ký tự.
- *Khác cách đặt dấu thanh*: cùng chữ và cùng dấu nhưng đặt dấu ở vị trí khác (`hoá` / `hóa`, `khoẻ` / `khỏe`). Đây là **hai cách viết đều hợp lệ**, không phải lỗi OCR.
- *Chỉ khớp khi bỏ hết dấu*: các chữ cái gốc giống nhau nhưng dấu khác. Đây mới là môn OCR đọc được chữ nhưng **sai dấu thật**.

**Giới hạn:** tên môn trên website có thể khác tên trong PDF (đổi tên, viết tắt, khóa học khác), nên "không tìm thấy" có thể do khác tên chứ không phải do OCR. Con số này là **cận trên** của lỗi OCR, không phải số đo chính xác.

In [7]:
subjects = Counter()
for r in recs:
    for c in r.get("curriculum") or []:
        for s in c["subjects"]:
            subjects[s.strip()] += 1

TONE_MARKS = {"̀", "́", "̃", "̉", "̣"}   # huyền, sắc, ngã, hỏi, nặng

def canon_tone(s):
    """Đưa dấu thanh về một vị trí chuẩn trong từng từ (cụm chữ cái liền nhau), để hoá = hóa.
    Xử lý theo cụm chữ cái, không theo khoảng trắng, để từ dính dấu câu (vd 'vụ;') vẫn khớp."""
    def one(m):
        d = unicodedata.normalize("NFD", m.group(0))
        return "".join(c for c in d if c not in TONE_MARKS) + "".join(c for c in d if c in TONE_MARKS)
    return re.sub(r"[^\W\d_]+", one, norm_ws(s).lower())

corpus = norm_ws(" ".join(files["text"])).lower()
corpus_ct = canon_tone(corpus)
corpus_nd = strip_marks(corpus)

exact, variant, nodiac, missing = [], [], [], []
for s in subjects:
    q = norm_ws(s).lower()
    if q in corpus:
        exact.append(s)
    elif canon_tone(q) in corpus_ct:
        variant.append(s)
    elif strip_marks(q) in corpus_nd:
        nodiac.append(s)
    else:
        missing.append(s)

N = len(subjects)
print("1. TỔNG QUAN")
print(f"   Tên môn học khác nhau trên website: {N} (từ {sum(1 for r in recs if r.get('curriculum'))} trang ngành)")
print()
print("2. LỖI / VẤN ĐỀ")
print(f"   Khớp nguyên văn                          : {len(exact):>4} ({100 * len(exact) / N:.1f}%)")
print(f"   Khác cách đặt dấu thanh (hợp lệ, không phải lỗi): {len(variant):>4} ({100 * len(variant) / N:.1f}%)")
print(f"   Chỉ khớp khi bỏ hết dấu (OCR sai dấu thật): {len(nodiac):>4} ({100 * len(nodiac) / N:.1f}%)")
print(f"   Không tìm thấy                           : {len(missing):>4} ({100 * len(missing) / N:.1f}%)")
print()
print("   Ví dụ khác cách đặt dấu thanh (hợp lệ):", variant[:3])
print()
print("   Ví dụ môn OCR sai dấu thật (tên trên web  →  đoạn tương ứng trong OCR):")
for s in nodiac[:6]:
    q = strip_marks(norm_ws(s).lower())
    i = corpus_nd.find(q)
    print(f"   {s!r}  →  {corpus[i:i + len(q)]!r}")
print()
print("   Ví dụ môn không tìm thấy (có thể do khác tên, không chắc là lỗi OCR):")
print("   ", missing[:8])
print()
print("3. TỔNG HỢP")
print(f"   Có thể có lỗi OCR: {len(nodiac)} môn sai dấu + tối đa {len(missing)} môn không tìm thấy")
add_issue("5. Đối chiếu tên môn", "tên môn học chỉ khớp khi bỏ hết dấu (OCR sai dấu thật)", len(nodiac), "tên môn", "sửa theo tên môn trên website (nguồn đúng)")
add_issue("5. Đối chiếu tên môn", "tên môn học không tìm thấy trong chữ OCR (cận trên, có thể do khác tên)", len(missing), "tên môn", "kiểm tra thủ công mẫu: khác tên hay OCR bỏ sót")

1. TỔNG QUAN
   Tên môn học khác nhau trên website: 640 (từ 25 trang ngành)

2. LỖI / VẤN ĐỀ
   Khớp nguyên văn                          :  522 (81.6%)
   Khác cách đặt dấu thanh (hợp lệ, không phải lỗi):    3 (0.5%)
   Chỉ khớp khi bỏ hết dấu (OCR sai dấu thật):   19 (3.0%)
   Không tìm thấy                           :   96 (15.0%)

   Ví dụ khác cách đặt dấu thanh (hợp lệ): ['Hoá - Hoá sinh', 'Giáo dục và tư vấn sức khoẻ', 'Mô hình hóa quy trình nghiệp vụ']

   Ví dụ môn OCR sai dấu thật (tên trên web  →  đoạn tương ứng trong OCR):
   'Tâm lý y học - Y đức'  →  'tâm lý y học - ÿ đức'
   'Tổ chức hạch toán kế toán'  →  'tô chức hạch toán kế toán'
   'Quyền con người và trách nhiệm xã hội của doanh nghiệp'  →  'quyên con người và trách nhiệm xã hội của doanh nghiệp'
   'Tiếng Anh chuyên ngành xuất nhập khẩu và logistics'  →  'tiếng anh chuyên ngành xuất nhập khâu và logistics'
   'Thiết kế chương trình và chương trình chi tiết'  →  'thiết kế chương trình và chương trình chỉ tiệt'
   'B

**Giải thích:** vì danh sách tên môn là nguồn đúng, đây là cách rẻ nhất để vừa đo vừa **sửa** lỗi dấu: thay đoạn OCR sai dấu bằng tên môn chuẩn từ website.

**4. Hướng giải quyết:** (1) dùng danh sách tên môn làm từ điển sửa: tìm đoạn OCR khớp bỏ dấu rồi thay bằng tên chuẩn; (2) với môn không tìm thấy, lấy mẫu khoảng 20 môn so với PDF để biết là khác tên hay OCR bỏ sót; (3) ghi rõ trong báo cáo đây là cận trên.

## 6. Trang có dạng bảng

**Cách nhận biết:** OCR đọc bảng theo từng ô nên các dòng bị ngắn và rời rạc. Trang có ít nhất 100 từ mà **hơn 60% số dòng ngắn hơn 20 ký tự** được coi là *trang dạng bảng*. Đây là quy tắc ước lượng, không phải nhận dạng bảng thật; có thể nhầm trang danh sách ngắn thành bảng.

In [8]:
def short_ratio(t):
    ls = [l.strip() for l in t.split("\n") if l.strip()]
    return sum(1 for l in ls if len(l) < 20) / len(ls) if ls else 0.0

pages["short_ratio"] = pages["text"].map(short_ratio)
pages["median_line"] = pages["text"].map(lambda t: pd.Series([len(l.strip()) for l in t.split("\n") if l.strip()] or [0]).median())
dense = pages[pages["n_words"] >= 100]
tab = dense[dense["short_ratio"] > 0.6]

print("1. TỔNG QUAN: trang có ít nhất 100 từ:", len(dense))
print("   Độ dài dòng trung vị (ký tự) — trang văn xuôi:", int(dense[dense['short_ratio'] <= 0.6]['median_line'].median()),
      "| trang dạng bảng:", int(tab['median_line'].median()) if len(tab) else "-")
print()
print("2. LỖI / VẤN ĐỀ: trang dạng bảng")
print(f"   {len(tab)} trang ({100 * len(tab) / len(dense):.1f}% trang có ≥ 100 từ)")
print(f"   Trong đó đã tự xoay: {int((tab['rotate'] != 0).sum())} trang ({100 * (tab['rotate'] != 0).mean():.0f}%)")
print()
ex_row = tab.iloc[len(tab) // 2]
print(f"   Ví dụ (bản ghi {ex_row['record_id']}, trang {ex_row['page']}): 12 dòng đầu")
for l in [l.strip() for l in ex_row["text"].split("\n") if l.strip()][:12]:
    print("     ", repr(l[:70]))
print()
print("3. TỔNG HỢP")
print(tab.groupby("record_id").size().rename("số trang dạng bảng").to_frame().to_string())
add_issue("6. Trang dạng bảng", "trang dạng bảng (≥ 100 từ, > 60% dòng ngắn): cấu trúc bảng mất khi OCR", len(tab), "trang", "dựng lại bảng từ vị trí chữ (image_to_data) hoặc dùng dữ liệu website thay cho bảng scan")

1. TỔNG QUAN: trang có ít nhất 100 từ: 1925
   Độ dài dòng trung vị (ký tự) — trang văn xuôi: 39 | trang dạng bảng: 12

2. LỖI / VẤN ĐỀ: trang dạng bảng
   194 trang (10.1% trang có ≥ 100 từ)
   Trong đó đã tự xoay: 140 trang (72%)

   Ví dụ (bản ghi tb-2235, trang 11): 12 dòng đầu
      'không cho hàng tồn kho'
      'cho doanh nghiệp.'
      '4.Phương pháp: Thảo'
      'luận nhóm'
      '- GV chỉ định nhóm'
      'trình bày nội dung đã'
      'phân công'
      'Chương 5:'
      'Quản trị tài sản'
      'cố định'
      '5.1. Quyết định'
      'thay thế tài sản cố'

3. TỔNG HỢP
           số trang dạng bảng
record_id                    
tb-2234                   117
tb-2235                    77


**Giải thích:** bảng trong bản scan mất cấu trúc: không còn biết ô nào thuộc hàng, cột nào. Điều này khác với bảng trong PDF có lớp chữ (đã được chuyển thành dòng `ô | ô | ô`).

**4. Hướng giải quyết:** có hai hướng. (1) Dựng lại bảng từ tọa độ từng từ mà Tesseract cung cấp (nhóm các từ theo hàng, cột). (2) Với danh sách môn học, dùng dữ liệu `curriculum` trên website thay cho bảng scan, vì nguồn này đã có cấu trúc.

## 7. Trùng lặp

In [9]:
print("1. TỔNG QUAN: so sánh nội dung các file OCR sau khi bỏ khoảng trắng thừa và chữ hoa / thường")
files["hash"] = files["text"].map(lambda t: hashlib.md5(norm_ws(t).lower().encode("utf-8")).hexdigest())
dup = files[files.duplicated("hash", keep=False)].sort_values("hash")
print()
print("2. LỖI / VẤN ĐỀ")
print(f"   File có nội dung giống hệt file khác: {len(dup)}")
if len(dup):
    show(dup[["record_id", "name", "n_pages", "hash"]].assign(name=dup["name"].str.slice(0, 50), hash=dup["hash"].str.slice(0, 8)))
print()
print("   Trang trùng nhau (cùng nội dung, trang có ≥ 50 từ):")
pg = pages[pages["n_words"] >= 50].copy()
pg["h"] = pg["text"].map(lambda t: hashlib.md5(norm_ws(t).lower().encode("utf-8")).hexdigest())
dp = pg[pg.duplicated("h", keep=False)]
print(f"   {len(dp)} trang thuộc {dp['h'].nunique()} nhóm")
print()
print("3. TỔNG HỢP")
add_issue("7. Trùng lặp", "file OCR có nội dung giống hệt file khác", len(dup), "file", "giữ một bản")
add_issue("7. Trùng lặp", "trang trùng nhau giữa các file", len(dp), "trang", "kiểm tra: trang chung (bìa, mẫu) thì giữ một bản; trang lặp ngoài ý muốn thì bỏ")

1. TỔNG QUAN: so sánh nội dung các file OCR sau khi bỏ khoảng trắng thừa và chữ hoa / thường

2. LỖI / VẤN ĐỀ
   File có nội dung giống hệt file khác: 4
   record_id                                          name  n_pages      hash
25   tb-2234  1. QĐ_Ban_hanh_CTĐT_Trinh_Do_Dai_Hoc_K39.pdf        2  b372fa71
27   tb-2234                      Legacy embedded document        2  b372fa71
12   tb-2235  QĐ_Dieu_Chinh_Bo_Sung_ĐCCTHP_CTĐT_ĐH_K39.pdf        2  d393057e
31   tb-2235                      Legacy embedded document        2  d393057e

   Trang trùng nhau (cùng nội dung, trang có ≥ 50 từ):
   8 trang thuộc 4 nhóm

3. TỔNG HỢP


**4. Hướng giải quyết:** file trùng hoàn toàn thì giữ một bản. Trang trùng thường là trang mẫu dùng chung giữa nhiều tài liệu (ví dụ phần mở đầu quyết định); nên giữ nhưng gắn nhãn để lúc truy vấn không trả về cùng một đoạn nhiều lần.

## 8. Tổng hợp vấn đề và hướng giải quyết

In [10]:
summary = pd.DataFrame(issues).sort_values("số lượng", ascending=False).reset_index(drop=True)
summary["có vấn đề"] = summary["số lượng"] > 0
print("A. CÁC KIỂM TRA CÓ VẤN ĐỀ")
show(summary[summary["có vấn đề"]][["mục", "lỗi / vấn đề", "số lượng", "đơn vị", "hướng giải quyết"]])
print()
print("B. KIỂM TRA KHÔNG PHÁT HIỆN VẤN ĐỀ (kết quả = 0)")
for _, r in summary[~summary["có vấn đề"]].iterrows():
    print(f"   - [{r['mục']}] {r['lỗi / vấn đề']}")
if summary["có vấn đề"].all():
    print("   (không có)")

A. CÁC KIỂM TRA CÓ VẤN ĐỀ
                      mục                                                             lỗi / vấn đề  số lượng   đơn vị                                                                            hướng giải quyết
0              4. Từ vựng       lượt từ không có trong từ vựng chuẩn (gồm cả tên riêng, thuật ngữ)     45476  lượt từ              phân loại: sửa dấu theo gợi ý; còn lại đối chiếu danh sách tên môn / tên riêng
1  3. Ký tự lạ / dòng rác                 dòng rác (≤ 2 ký tự không có số, hoặc < 50% là chữ / số)      4600     dòng                                         xóa dòng rác khi chuẩn hóa; giữ dòng ngắn chỉ có số
2              4. Từ vựng               trong đó có gợi ý đổi dấu (chưa duyệt; một phần gợi ý sai)      4473  lượt từ                       duyệt tay danh sách gợi ý rồi mới áp dụng; ưu tiên từ xuất hiện nhiều
3  3. Ký tự lạ / dòng rác                                 ký tự lạ (ngoài chữ, số, dấu câu thường)      3750    ký tự                 

### Thứ tự xử lý đề xuất

1. **Sửa dấu bằng nguồn đúng** (mục 5, 4): dùng tên môn học trên website và từ điển sửa lỗi đã duyệt tay.
2. **Xóa ký tự lạ và dòng rác** (mục 3): quy tắc đơn giản, rà danh sách ký hiệu có nghĩa trước.
3. **Xử lý trang ít chữ** (mục 2): xem ảnh một lần, loại trang bìa / trang trắng.
4. **Bảng trong bản scan** (mục 6): chọn hướng dựng lại bảng hoặc dùng dữ liệu website.
5. **Trùng lặp** (mục 7): gắn nhãn, không xóa vội.

**Giới hạn chung của phân tích này:** không có bản chữ đúng của các file scan, nên mọi con số ở trên là *dấu hiệu gián tiếp* hoặc *cận trên* của lỗi OCR, không phải tỷ lệ lỗi thật. Muốn có tỷ lệ lỗi thật cần sửa tay một mẫu trang làm chuẩn rồi đo.